In [9]:
# Motor Learning Analysis Pipeline - Properly Structured
# This breaks down your large code block into proper modules

# ============================================================================== 
# PART 1: IMPORTS AND CONFIGURATION
# ==============================================================================

import json
import pickle
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Callable, Union
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, gaussian_kde
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import pingouin as pg

import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# PART 2: CONFIGURATION CLASSES
# ==============================================================================

@dataclass
class PlotConfig:
    """Centralized plotting configuration that can be modified after creation."""
    
    # Color schemes
    colors: Dict[str, str] = field(default_factory=lambda: {
        'primary': '#667eea',
        'secondary': '#764ba2',
        'success': '#28a745',
        'warning': '#ffc107',
        'danger': '#dc3545',
        'vis1': '#1f77b4',
        'invis': '#ff7f0e',
        'vis2': '#2ca02c',
        'max_target': '#2E8B57',
        'min_target': '#B22222',
        'age_young': '#4CAF50',
        'age_middle': '#FF9800',
        'age_old': '#9C27B0'
    })
    
    # Figure settings
    figure_size_small: Tuple[int, int] = (8, 6)
    figure_size_medium: Tuple[int, int] = (12, 8)
    figure_size_large: Tuple[int, int] = (16, 12)
    figure_size_wide: Tuple[int, int] = (18, 6)
    figure_dpi: int = 300
    
    # Font settings
    font_size_small: int = 8
    font_size_medium: int = 10
    font_size_large: int = 12
    font_size_title: int = 14
    font_size_suptitle: int = 16
    font_weight_normal: str = 'normal'
    font_weight_bold: str = 'bold'
    
    # Plot styling
    alpha_scatter: float = 0.7
    alpha_fill: float = 0.3
    alpha_line: float = 0.8
    line_width_thin: float = 1
    line_width_medium: float = 2
    line_width_thick: float = 3
    marker_size_small: int = 40
    marker_size_medium: int = 60
    marker_size_large: int = 100
    
    # Grid and layout
    grid_alpha: float = 0.3
    tight_layout_pad: float = 2.0
    subplot_wspace: float = 0.3
    subplot_hspace: float = 0.4
    
    # Statistical visualization
    trendline_color: str = 'red'
    trendline_style: str = '--'
    trendline_alpha: float = 0.8
    
    # Age grouping colors
    age_group_colors: List[str] = field(default_factory=lambda: [
        '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#2196F3', '#795548'
    ])
    
    def update_colors(self, new_colors: Dict[str, str]):
        """Update color scheme."""
        self.colors.update(new_colors)
    
    def update_figure_sizes(self, **kwargs):
        """Update figure size settings."""
        for key, value in kwargs.items():
            if hasattr(self, f'figure_size_{key}'):
                setattr(self, f'figure_size_{key}', value)


@dataclass
class AnalysisConfig:
    """Enhanced analysis configuration with mutable plotting config."""
    
    # Directories
    base_output_dir: Path = Path('motor_learning_output')
    
    # Analysis thresholds
    min_complete_strides: int = 20
    motor_noise_strides: int = 20
    motor_noise_threshold: float = 0.3
    success_rate_threshold: float = 0.68
    target_size_threshold: float = 0.31
    max_strides_threshold: int = 415
    alpha_level: float = 0.05
    
    # Age binning
    age_bins: List[int] = field(default_factory=lambda: [7, 10, 13, 16, 18])
    age_labels: List[str] = field(default_factory=lambda: ['7-10', '10-13', '13-16', '16-18'])
    
    # Trial mappings
    trial_type_mapping: Dict[str, str] = field(default_factory=lambda: {
        'primer': 'vis1',
        'trial': 'invis',
        'vis': 'vis2',
        'pref': 'pref'
    })
    
    # Plotting configuration
    plot_config: PlotConfig = field(default_factory=PlotConfig)
    
    def __post_init__(self):
        """Create directory structure."""
        self.figures_dir = self.base_output_dir / 'figures'
        self.individual_plots_dir = self.figures_dir / 'individual_plots'
        self.population_plots_dir = self.figures_dir / 'population_plots'
        self.statistical_plots_dir = self.figures_dir / 'statistical_plots'
        self.reports_dir = self.base_output_dir / 'reports'
        self.exports_dir = self.base_output_dir / 'exports'
        self.processed_data_dir = self.base_output_dir / 'processed_data'
        
        # Create all directories
        for directory in [self.figures_dir, self.individual_plots_dir, 
                         self.population_plots_dir, self.statistical_plots_dir,
                         self.reports_dir, self.exports_dir, self.processed_data_dir]:
            directory.mkdir(parents=True, exist_ok=True)
        
        self.processed_data_file = self.processed_data_dir / 'processed_data.pkl'
    
    def update_plot_config(self, **kwargs):
        """Update plotting configuration."""
        for key, value in kwargs.items():
            if hasattr(self.plot_config, key):
                setattr(self.plot_config, key, value)


# ==============================================================================
# PART 3: BASE CLASSES
# ==============================================================================

class BaseProcessor(ABC):
    """Base class for all data processors."""
    
    def __init__(self, config: AnalysisConfig, debug: bool = True):
        self.config = config
        self.debug = debug
    
    def log(self, message: str, level: str = "info"):
        """Logging helper."""
        if self.debug:
            symbols = {"info": "📊", "warning": "⚠️", "error": "❌", "success": "✓"}
            print(f"{symbols.get(level, '•')} {message}")


class BaseVisualizer(ABC):
    """Base class for all visualizers."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.plot_config = config.plot_config
        self._setup_matplotlib_style()
    
    def _setup_matplotlib_style(self):
        """Set up matplotlib style based on config."""
        plt.rcParams.update({
            'font.size': self.plot_config.font_size_medium,
            'axes.titlesize': self.plot_config.font_size_title,
            'axes.labelsize': self.plot_config.font_size_medium,
            'xtick.labelsize': self.plot_config.font_size_small,
            'ytick.labelsize': self.plot_config.font_size_small,
            'legend.fontsize': self.plot_config.font_size_small,
            'figure.titlesize': self.plot_config.font_size_suptitle,
            'figure.dpi': self.plot_config.figure_dpi,
            'savefig.dpi': self.plot_config.figure_dpi,
            'axes.grid': True,
            'grid.alpha': self.plot_config.grid_alpha
        })
    
    def save_figure(self, fig: plt.Figure, filename: str, subdir: str = 'general'):
        """Save figure to appropriate directory."""
        if subdir == 'individual':
            save_path = self.config.individual_plots_dir / filename
        elif subdir == 'population':
            save_path = self.config.population_plots_dir / filename
        elif subdir == 'statistical':
            save_path = self.config.statistical_plots_dir / filename
        else:
            save_path = self.config.figures_dir / filename
        
        fig.savefig(save_path, 
                   dpi=self.plot_config.figure_dpi, 
                   bbox_inches='tight',
                   facecolor='white',
                   edgecolor='none')
        plt.close(fig)
        return save_path
    
    def add_trendline(self, ax, x, y, show_stats: bool = True, **kwargs):
        """Add trendline with correlation statistics."""
        x_clean = pd.to_numeric(x, errors='coerce')
        y_clean = pd.to_numeric(y, errors='coerce')
        valid = x_clean.notna() & y_clean.notna()
        
        if valid.sum() < 2:
            return
        
        x_vals = x_clean[valid]
        y_vals = y_clean[valid]
        
        try:
            coeffs = np.polyfit(x_vals, y_vals, 1)
            trendline = np.poly1d(coeffs)
            r, p = pearsonr(x_vals, y_vals)
            
            # Use config or override values
            color = kwargs.get('color', self.plot_config.trendline_color)
            linestyle = kwargs.get('linestyle', self.plot_config.trendline_style)
            alpha = kwargs.get('alpha', self.plot_config.trendline_alpha)
            
            ax.plot(x_vals, trendline(x_vals), 
                   color=color, linestyle=linestyle, alpha=alpha,
                   linewidth=self.plot_config.line_width_medium)
            
            if show_stats:
                stats_text = kwargs.get('stats_format', 'r² = {r2:.3f}\\np = {p:.3f}\\nn = {n}')
                ax.text(0.05, 0.95, stats_text.format(r2=r**2, p=p, n=len(x_vals)), 
                       transform=ax.transAxes,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                       verticalalignment='top', 
                       fontsize=self.plot_config.font_size_small)
        except Exception:
            pass


# ==============================================================================
# PART 4: DATA STRUCTURES
# ==============================================================================

@dataclass
class SubjectData:
    """Container for individual subject data."""
    subject_id: str
    metadata: Dict[str, Any]
    trial_data: Dict[str, Dict[str, Any]]
    
    @property
    def age(self) -> float:
        """Get age in years."""
        return self.metadata.get('age_months', np.nan) / 12


@dataclass
class TrialData:
    """Container for trial data."""
    data: pd.DataFrame
    anomalies: Dict[int, list]
    trial_type: str
    
    @property
    def is_valid(self) -> bool:
        """Check if trial data is valid."""
        return self.data is not None and not self.data.empty


# ==============================================================================
# PART 5: DATA VALIDATION AND PROCESSING
# ==============================================================================

class DataValidator:
    """Handles data validation and anomaly detection."""
    
    @staticmethod
    def validate_dataframe(df: pd.DataFrame, required_cols: List[str] = None) -> bool:
        """Validate dataframe has required columns and data."""
        if df is None or df.empty:
            return False
        if required_cols and not all(col in df.columns for col in required_cols):
            return False
        return True
    
    @staticmethod
    def detect_anomalies(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Detect and flag anomalies in stride data."""
        if df is None or df.empty:
            return df, {}
        
        df = df.copy()
        df['Anomalous'] = False
        anomalies = {}
        
        # Time-based anomalies
        time_col = next((col for col in ['Time', 'Timestamp', 'Time (s)'] 
                        if col in df.columns), None)
        if time_col:
            df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
            time_diff = df[time_col].diff()
            jump_mask = time_diff > time_diff.quantile(0.99) * 5
            
            for idx in df.index[jump_mask.fillna(False)]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('time_jump')
        
        # Sum of gains and steps anomalies
        if 'Sum of gains and steps' in df.columns:
            high_mask = df['Sum of gains and steps'] > 4
            zero_mask = df['Sum of gains and steps'] == 0
            
            for idx in df.index[high_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_high')
            
            for idx in df.index[zero_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_zero')
        
        return df, anomalies


class FileLoader:
    """Handles file loading operations."""
    
    @staticmethod
    def load_file(file_path) -> Optional[pd.DataFrame]:
        """Load and validate a single data file."""
        try:
            df = pd.read_csv(file_path, sep='\\t')
            
            # Basic cleaning
            if 'Stride Number' in df.columns:
                df['Stride Number'] = pd.to_numeric(df['Stride Number'], errors='coerce')
                df = df.dropna(subset=['Stride Number'])
                df = df.drop_duplicates(subset=['Stride Number'])
                df = df.sort_values('Stride Number')
            
            return df if not df.empty else None
            
        except Exception:
            return None


class TrialProcessor(BaseProcessor):
    """Processes trial data files."""
    
    def process_trial_files(self, subject_dir: Path, trial_prefix: str) -> Optional[pd.DataFrame]:
        """Find and combine trial files for a given trial type."""
        all_files = sorted(subject_dir.glob(f"{trial_prefix}*.txt"))
        
        if not all_files:
            return None
        
        # Special handling for preference trials
        if trial_prefix == 'pref':
            largest_file = max(all_files, key=lambda f: f.stat().st_size)
            return FileLoader.load_file(largest_file)
        
        # Single file case
        if len(all_files) == 1:
            return FileLoader.load_file(all_files[0])
        
        # Multiple files - combine them
        return self._combine_trial_fragments(all_files)
    
    def _combine_trial_fragments(self, files: List[Path]) -> Optional[pd.DataFrame]:
        """Combine multiple trial fragments intelligently."""
        dfs = []
        for f in files:
            df = FileLoader.load_file(f)
            if df is not None:
                dfs.append(df)
        
        if not dfs:
            return None
        
        combined = pd.concat(dfs, ignore_index=True)
        
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number')
            combined = combined.drop_duplicates('Stride Number')
        
        return combined


# ==============================================================================
# PART 6: METRICS CALCULATION
# ==============================================================================

class MetricsEngine:
    """Core metrics calculation engine."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
    
    def calculate_period_metrics(self, period_data: pd.DataFrame, 
                               trial_type: str, condition: str) -> Dict:
        """Calculate all metrics for a specific period."""
        metrics = {}
        
        # Success rate
        metrics[f'{trial_type}_sr_{condition}_const'] = period_data['Success'].mean()
        
        # Stride metrics
        if 'Sum of gains and steps' in period_data.columns:
            sogs = period_data['Sum of gains and steps']
            metrics[f'{trial_type}_sd_{condition}_const'] = sogs.std()
            metrics[f'{trial_type}_msl_{condition}_const'] = sogs.mean()
            
            if 'Constant' in period_data.columns:
                metrics[f'{trial_type}_error_{condition}_const'] = (sogs - period_data['Constant']).mean()
        
        # Asymmetry
        if all(col in period_data.columns for col in ['Right step length', 'Left step length']):
            asymmetry = self._calculate_asymmetry(
                period_data['Right step length'], 
                period_data['Left step length']
            )
            if asymmetry is not None:
                metrics[f'{trial_type}_asymmetry_{condition}_const'] = asymmetry
        
        # Strides between successes
        strides_between = self._calculate_strides_between_successes(period_data)
        if strides_between is not None:
            metrics[f'{trial_type}_strides_between_success_{condition}_const'] = strides_between
        
        return metrics
    
    def calculate_preference_metrics(self, pref_df: pd.DataFrame) -> Dict:
        """Calculate metrics from preference trial."""
        metrics = {'mot_noise': None, 'pref_asymmetry': None}
        
        if pref_df is None or pref_df.empty:
            return metrics
        
        if not all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
            return metrics
        
        # Get non-zero steps
        right_steps = pref_df['Right step length']
        left_steps = pref_df['Left step length']
        
        right_clean = right_steps[(right_steps != 0) & (right_steps.notna())]
        left_clean = left_steps[(left_steps != 0) & (left_steps.notna())]
        
        min_required = self.config.motor_noise_strides
        if len(right_clean) < min_required or len(left_clean) < min_required:
            return metrics
        
        # Calculate motor noise
        final_right = right_clean.iloc[-1]
        final_left = left_clean.iloc[-1]
        
        if final_right <= 0 or final_left <= 0:
            return metrics
        
        # Normalize steps
        norm_right = right_clean / final_right
        norm_left = left_clean / final_left
        
        min_length = min(len(norm_right), len(norm_left))
        if min_length < min_required:
            return metrics
        
        sum_steps = norm_right.iloc[:min_length] + norm_left.iloc[:min_length]
        
        # Motor noise from last N points
        if len(sum_steps) >= self.config.motor_noise_strides:
            noise = sum_steps.tail(self.config.motor_noise_strides).std()
            if not pd.isna(noise) and noise > 0:
                metrics['mot_noise'] = noise
        
        # Calculate asymmetry
        if len(right_clean) >= self.config.motor_noise_strides and len(left_clean) >= self.config.motor_noise_strides:
            last_n_right = right_clean.tail(self.config.motor_noise_strides) / final_right
            last_n_left = left_clean.tail(self.config.motor_noise_strides) / final_left
            
            asymmetry = self._calculate_asymmetry(last_n_right.values, last_n_left.values)
            if asymmetry is not None:
                metrics['pref_asymmetry'] = asymmetry
        
        return metrics
    
    def _calculate_asymmetry(self, right_values, left_values) -> Optional[float]:
        """Calculate step length asymmetry."""
        denominator = right_values + left_values
        valid_mask = denominator != 0
        
        if not valid_mask.any():
            return None
        
        asymmetry_vals = np.abs((right_values - left_values) / denominator)[valid_mask]
        return np.mean(asymmetry_vals) if len(asymmetry_vals) > 0 else None
    
    def _calculate_strides_between_successes(self, df: pd.DataFrame) -> Optional[float]:
        """Calculate average strides between successful trials."""
        if df is None or 'Success' not in df.columns:
            return None
        
        df = df.reset_index(drop=True)
        success_positions = df.index[df['Success'] == 1].tolist()
        
        if len(success_positions) < 2:
            return None
        
        return np.mean(np.diff(success_positions))


# ==============================================================================
# PART 7: CONTINUATION POINT
# ==============================================================================

# This is where you would continue with:
# - DataManager class
# - StatisticalAnalyzer class (the one with the indentation error)
# - Visualization classes
# - Main analysis interface

print("✅ Motor Learning Analysis structure loaded successfully!")
print("📝 This provides the foundation classes and configuration.")
print("🔧 Continue by adding the DataManager and StatisticalAnalyzer classes.")

# Motor Learning Analysis Pipeline - Continuation
# This contains the DataManager and StatisticalAnalyzer classes

# ==============================================================================
# PART 8: DATA MANAGER
# ==============================================================================

class DataManager(BaseProcessor):
    """Manages all data loading and processing operations."""
    
    def __init__(self, metadata_path: str, data_root_dir: str, 
                 config: AnalysisConfig, force_reprocess: bool = False, 
                 debug: bool = True):
        super().__init__(config, debug)
        self.metadata_path = metadata_path
        self.data_root_dir = Path(data_root_dir)
        self.trial_processor = TrialProcessor(config, debug)
        self.metrics_engine = MetricsEngine(config)
        
        self.subjects: Dict[str, SubjectData] = {}
        self.metadata: pd.DataFrame = None
        
        if not force_reprocess and config.processed_data_file.exists():
            self._load_processed_data()
        else:
            self._process_all_data()
            self._save_processed_data()
    
    def _load_processed_data(self):
        """Load previously processed data."""
        try:
            with open(self.config.processed_data_file, 'rb') as f:
                saved_data = pickle.load(f)
                
            # Convert to SubjectData objects
            for subject_id, data in saved_data.items():
                self.subjects[subject_id] = SubjectData(
                    subject_id=subject_id,
                    metadata=data['metadata'],
                    trial_data=data['trial_data']
                )
            
            # Rebuild metadata DataFrame
            self.metadata = pd.DataFrame.from_dict(
                {subj: data.metadata for subj, data in self.subjects.items()}, 
                orient='index'
            )
            self.log(f"Loaded {len(self.subjects)} subjects from cache", "success")
        except Exception as e:
            self.log(f"Failed to load cached data: {e}", "warning")
            self._process_all_data()
            self._save_processed_data()
    
    def _save_processed_data(self):
        """Save processed data."""
        try:
            # Convert to serializable format
            save_data = {}
            for subject_id, subject in self.subjects.items():
                save_data[subject_id] = {
                    'metadata': subject.metadata,
                    'trial_data': subject.trial_data
                }
            
            with open(self.config.processed_data_file, 'wb') as f:
                pickle.dump(save_data, f)
            self.log(f"Saved processed data to {self.config.processed_data_file}", "success")
        except Exception as e:
            self.log(f"Failed to save processed data: {e}", "error")
    
    def _process_all_data(self):
        """Process all subject data."""
        self._load_metadata()
        total_subjects = len(self.metadata)
        
        self.log(f"Processing {total_subjects} subjects...")
        
        for i, (_, row) in enumerate(self.metadata.iterrows(), 1):
            subject_id = row['ID']
            if self.debug and i % 10 == 0:
                self.log(f"Progress: {i}/{total_subjects}")
            
            subject_data = self._process_subject(subject_id, row)
            if subject_data:
                self.subjects[subject_id] = subject_data
    
    def _load_metadata(self):
        """Load and clean metadata."""
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata['DOB'] = pd.to_datetime(self.metadata['DOB'], errors='coerce')
        self.metadata['Session Date'] = pd.to_datetime(self.metadata['Session Date'], errors='coerce')
        self.metadata = self.metadata.dropna(subset=['ID', 'age_months'])
    
    def _process_subject(self, subject_id: str, metadata_row: pd.Series) -> Optional[SubjectData]:
        """Process data for a single subject."""
        subject_dir = self.data_root_dir / subject_id
        if not subject_dir.exists():
            return None
        
        trial_data = {}
        
        for original_type, mapped_type in self.config.trial_type_mapping.items():
            df = self.trial_processor.process_trial_files(subject_dir, original_type)
            
            if df is not None:
                # Process the data
                if original_type != 'pref':
                    df, anomalies = self._process_trial_data(df)
                else:
                    df = df.drop_duplicates(subset='Left heel strike', keep='last')
                    anomalies = {}
                
                trial_data[mapped_type] = {
                    'data': df,
                    'anomalies': anomalies
                }
        
        if not trial_data:
            return None
        
        return SubjectData(
            subject_id=subject_id,
            metadata=metadata_row.to_dict(),
            trial_data=trial_data
        )
    
    def _process_trial_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Process trial data and calculate derived metrics."""
        if df is None or df.empty:
            return None, {}
        
        # Validate required columns
        required_cols = ['Stride Number', 'Success', 'Upper bound success', 
                        'Lower bound success', 'Constant']
        if not all(col in df.columns for col in required_cols):
            return None, {}
        
        # Process trial data
        df = df.sort_values('Stride Number')
        df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        df = df.drop_duplicates(subset='Stride Number', keep='last')
        
        # Scale sum of gains and steps
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        # Detect anomalies
        df, anomalies = DataValidator.detect_anomalies(df)
        
        return df, anomalies
    
    def get_trial_data(self, subject_id: str, trial_type: str) -> Optional[pd.DataFrame]:
        """Get trial data for a specific subject and trial type."""
        if subject_id not in self.subjects:
            return None
        
        trial_dict = self.subjects[subject_id].trial_data.get(trial_type)
        return trial_dict['data'] if trial_dict else None
    
    def calculate_metrics(self) -> pd.DataFrame:
        """Calculate metrics for all subjects."""
        results = []
        
        self.log(f"Calculating metrics for {len(self.subjects)} subjects...")
        
        for subject_id, subject in self.subjects.items():
            result = self._calculate_subject_metrics(subject)
            if result:
                results.append(result)
        
        if not results:
            self.log("No valid metrics calculated!", level="error")
            return pd.DataFrame()
        
        df = pd.DataFrame(results).infer_objects()
        self.log(f"Successfully calculated metrics for {len(df)} subjects", level="success")
        return df
    
    def _calculate_subject_metrics(self, subject: SubjectData) -> Optional[Dict]:
        """Calculate metrics for a single subject."""
        result = {
            'ID': subject.subject_id,
            'age': subject.age,
            'session_date': subject.metadata.get('Session Date')
        }
        
        # Process each trial type
        for trial_type in ['vis1', 'invis', 'vis2']:
            trial_dict = subject.trial_data.get(trial_type)
            if not trial_dict:
                continue
            
            df = trial_dict['data']
            if df is None or df.empty or 'Success' not in df.columns:
                continue
            
            # Calculate metrics for both conditions
            for condition in ['max', 'min']:
                period_data, indices = self._get_period_data(df, condition)
                if period_data is not None and not period_data.empty:
                    metrics = self.metrics_engine.calculate_period_metrics(
                        period_data, trial_type, condition
                    )
                    result.update(metrics)
                    result[f'{trial_type}_{condition}_const_indices'] = indices
            
            # Add trial metadata
            if df is not None:
                result.update({
                    f'{trial_type}_min_target_size': df['Target size'].min() if 'Target size' in df.columns else None,
                    f'{trial_type}_max_constant': df['Constant'].max() if 'Constant' in df.columns else None,
                    f'{trial_type}_min_constant': df['Constant'].min() if 'Constant' in df.columns else None
                })
                
                # Order information for invis trials
                if trial_type == 'invis':
                    result.update(self._calculate_condition_order(df))
        
        # Process preference trial
        pref_dict = subject.trial_data.get('pref')
        if pref_dict:
            pref_metrics = self.metrics_engine.calculate_preference_metrics(pref_dict['data'])
            result.update(pref_metrics)
        
        return result
    
    def _get_period_data(self, df: pd.DataFrame, condition: str, 
                        length: int = 20) -> Tuple[Optional[pd.DataFrame], Optional[List]]:
        """Extract data for specific condition period."""
        if df is None or df.empty:
            return None, None
        
        if 'Target size' not in df.columns or 'Constant' not in df.columns:
            return None, None
        
        min_target = df['Target size'].min()
        target_tolerance = 0.001
        
        min_target_periods = df[df['Target size'] <= min_target + target_tolerance]
        if min_target_periods.empty:
            return None, None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max'
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[
            np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)
        ]
        
        if period_data.empty:
            return None, None
        
        return period_data.tail(length), period_data.index.tolist()
    
    def _calculate_condition_order(self, df: pd.DataFrame) -> Dict:
        """Determine which condition came first for invis trials."""
        all_max_indices = df.index[df['Constant'] == df['Constant'].max()].tolist()
        all_min_indices = df.index[df['Constant'] == df['Constant'].min()].tolist()
        
        if all_max_indices and all_min_indices:
            first_max = min(all_max_indices)
            first_min = min(all_min_indices)
            return {
                'invis_max_first': first_max < first_min,
                'invis_min_first': first_min < first_max
            }
        
        return {'invis_max_first': False, 'invis_min_first': False}
    
    def filter_subjects(self, max_target_size: float = None, 
                       min_age: float = None, max_age: float = None,
                       required_trial_types: List[str] = None) -> 'DataManager':
        """Create a filtered DataManager instance."""
        filtered_subjects = {}
        
        for subject_id, subject in self.subjects.items():
            # Age filtering
            age = subject.age
            if min_age is not None and age < min_age:
                continue
            if max_age is not None and age > max_age:
                continue
            
            # Check required trial types
            if required_trial_types:
                missing_trials = [
                    t for t in required_trial_types 
                    if t not in subject.trial_data or 
                       subject.trial_data[t]['data'] is None
                ]
                if missing_trials:
                    continue
            
            # Apply other filters
            valid = True
            for trial_type, trial_dict in subject.trial_data.items():
                if trial_dict and trial_dict['data'] is not None:
                    df = trial_dict['data']
                    
                    if (max_target_size is not None and 
                        'Target size' in df.columns and 
                        df['Target size'].min() > max_target_size):
                        valid = False
                        break
            
            if valid:
                filtered_subjects[subject_id] = subject
        
        # Create new instance with filtered data
        new_manager = DataManager.__new__(DataManager)
        new_manager.config = self.config
        new_manager.metadata_path = self.metadata_path
        new_manager.data_root_dir = self.data_root_dir
        new_manager.debug = self.debug
        new_manager.trial_processor = self.trial_processor
        new_manager.metrics_engine = self.metrics_engine
        new_manager.subjects = filtered_subjects
        new_manager.metadata = pd.DataFrame.from_dict(
            {subj: data.metadata for subj, data in filtered_subjects.items()}, 
            orient='index'
        )
        
        return new_manager


# ==============================================================================
# PART 9: STATISTICAL ANALYZER - PROPERLY STRUCTURED
# ==============================================================================

class StatisticalAnalyzer:
    """Handles all statistical analyses."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        self.config = config
        self.metrics_df = metrics_df
        
        # Apply motor noise filter
        if 'mot_noise' in metrics_df.columns:
            self.filtered_df = metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_df = metrics_df
    
    def run_regression_analysis(self, trial_type: str = 'invis', 
                               condition: str = 'max',
                               predictors: List[str] = None) -> Dict:
        """Run regression analysis."""
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
        
        available_predictors = [p for p in predictors if p in self.filtered_df.columns]
        target_col = f'{trial_type}_sr_{condition}_const'
        
        if target_col not in self.filtered_df.columns:
            raise ValueError(f"Target column {target_col} not found")
        
        # Prepare data
        valid_data = self.filtered_df[available_predictors + [target_col]].dropna()
        
        if len(valid_data) < 10:
            raise ValueError(f"Insufficient data: only {len(valid_data)} valid samples")
        
        X = valid_data[available_predictors]
        y = valid_data[target_col]
        
        # Split and train
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', LinearRegression())
        ])
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        return {
            'model': model,
            'metrics': {
                'r2': r2_score(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'n_samples': len(valid_data)
            },
            'feature_importances': dict(zip(available_predictors, 
                                          np.abs(model.named_steps['regressor'].coef_)))
        }
    
    def run_repeated_measures_anova(self) -> Dict:
        """Run repeated measures ANOVA with covariates using pingouin."""
        # Prepare long-format data
        long_rows = []
        
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'success_rate': row[col],
                                'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna()
        
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {}
        
        # Basic repeated measures ANOVA (without covariates)
        try:
            # Main effect of trial type
            if len(df_long['trial_type'].unique()) > 1:
                aov_trial = pg.rm_anova(data=df_long, dv='success_rate', 
                                       within='trial_type', subject='subject', detailed=True)
                results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size': float(aov_trial['ng2'].iloc[0])
                }
            
            # Main effect of condition
            if len(df_long['condition'].unique()) > 1:
                aov_condition = pg.rm_anova(data=df_long, dv='success_rate', 
                                           within='condition', subject='subject', detailed=True)
                results['condition_effect'] = {
                    'F': float(aov_condition['F'].iloc[0]),
                    'p_value': float(aov_condition['p-unc'].iloc[0]),
                    'effect_size': float(aov_condition['ng2'].iloc[0])
                }
            
            # Interaction effect
            if len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1:
                aov_interaction = pg.rm_anova(data=df_long, dv='success_rate', 
                                             within=['trial_type', 'condition'], 
                                             subject='subject', detailed=True)
                interaction_row = aov_interaction[aov_interaction['Source'].str.contains('trial_type \\* condition')]
                if not interaction_row.empty:
                    results['interaction_effect'] = {
                        'F': float(interaction_row['F'].iloc[0]),
                        'p_value': float(interaction_row['p-unc'].iloc[0]),
                        'effect_size': float(interaction_row['ng2'].iloc[0])
                    }
        except Exception as e:
            results['basic_anova_error'] = str(e)
        
        # ANCOVA with covariates
        try:
            # Check which covariates are available
            available_covariates = []
            if 'age' in df_long.columns and df_long['age'].notna().sum() > 0:
                available_covariates.append('age')
            if 'motor_noise' in df_long.columns and df_long['motor_noise'].notna().sum() > 0:
                available_covariates.append('motor_noise')
            if 'pref_asymmetry' in df_long.columns and df_long['pref_asymmetry'].notna().sum() > 0:
                available_covariates.append('pref_asymmetry')
            
            if available_covariates:
                results['ancova_covariates'] = available_covariates
                
                # Covariate correlations
                for cov in available_covariates:
                    cov_corr = df_long.groupby('subject').agg({
                        'success_rate': 'mean',
                        cov: 'first'
                    }).reset_index()
                    if len(cov_corr) > 5:
                        r, p = pearsonr(cov_corr[cov], cov_corr['success_rate'])
                        results[f'{cov}_covariate_effect'] = {
                            'correlation': float(r),
                            'p_value': float(p),
                            'interpretation': f'{cov} effect on overall success rate'
                        }
                
                # Mixed effects analysis using statsmodels for proper ANCOVA
                try:
                    import statsmodels.api as sm
                    from statsmodels.formula.api import mixedlm
                    
                    # Prepare formula with available covariates
                    covariate_terms = ' + '.join(available_covariates)
                    formula = f"success_rate ~ trial_type * condition + {covariate_terms}"
                    
                    # Fit mixed effects model
                    model = mixedlm(formula, df_long, groups=df_long['subject'])
                    fitted_model = model.fit()
                    
                    # Extract covariate effects
                    covariate_effects = {}
                    for cov in available_covariates:
                        if cov in fitted_model.params.index:
                            covariate_effects[f'{cov}_effect'] = {
                                'coefficient': float(fitted_model.params[cov]),
                                'p_value': float(fitted_model.pvalues[cov]),
                                'confidence_interval': [float(fitted_model.conf_int().loc[cov, 0]), 
                                                      float(fitted_model.conf_int().loc[cov, 1])],
                                'interpretation': f'Change in success rate per unit increase in {cov}'
                            }
                    
                    results['mixed_effects_covariates'] = covariate_effects
                    results['mixed_effects_model_summary'] = {
                        'aic': float(fitted_model.aic),
                        'bic': float(fitted_model.bic),
                        'log_likelihood': float(fitted_model.llf)
                    }
                    
                except Exception as e:
                    results['mixed_effects_error'] = str(e)
            
        except Exception as e:
            results['ancova_error'] = str(e)
        
        return results
    
    def run_repeated_measures_anova_msl(self) -> Dict:
        """Run repeated measures ANOVA for mean stride length with covariates using pingouin."""
        # Prepare long-format data
        long_rows = []
        
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if '_msl_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'mean_stride_length': row[col],
                                'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna()
        
        if df_long.empty:
            return {'error': 'No valid mean stride length data for analysis'}
        
        results = {}
        
        # Basic repeated measures ANOVA (without covariates)
        try:
            # Main effect of trial type
            if len(df_long['trial_type'].unique()) > 1:
                aov_trial = pg.rm_anova(data=df_long, dv='mean_stride_length', 
                                       within='trial_type', subject='subject', detailed=True)
                results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size': float(aov_trial['ng2'].iloc[0])
                }
            
            # Main effect of condition
            if len(df_long['condition'].unique()) > 1:
                aov_condition = pg.rm_anova(data=df_long, dv='mean_stride_length', 
                                           within='condition', subject='subject', detailed=True)
                results['condition_effect'] = {
                    'F': float(aov_condition['F'].iloc[0]),
                    'p_value': float(aov_condition['p-unc'].iloc[0]),
                    'effect_size': float(aov_condition['ng2'].iloc[0])
                }
            
            # Interaction effect
            if len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1:
                aov_interaction = pg.rm_anova(data=df_long, dv='mean_stride_length', 
                                             within=['trial_type', 'condition'], 
                                             subject='subject', detailed=True)
                interaction_row = aov_interaction[aov_interaction['Source'].str.contains('trial_type \\* condition')]
                if not interaction_row.empty:
                    results['interaction_effect'] = {
                        'F': float(interaction_row['F'].iloc[0]),
                        'p_value': float(interaction_row['p-unc'].iloc[0]),
                        'effect_size': float(interaction_row['ng2'].iloc[0])
                    }
        except Exception as e:
            results['basic_anova_error'] = str(e)
        
        # Add descriptive statistics
        results['descriptive_stats'] = {
            'n_subjects': len(df_long['subject'].unique()),
            'n_observations': len(df_long),
            'mean_stride_length_overall': float(df_long['mean_stride_length'].mean()),
            'std_stride_length_overall': float(df_long['mean_stride_length'].std()),
            'by_trial_type': df_long.groupby('trial_type')['mean_stride_length'].agg(['mean', 'std', 'count']).to_dict(),
            'by_condition': df_long.groupby('condition')['mean_stride_length'].agg(['mean', 'std', 'count']).to_dict()
        }
        
        return results

    def run_repeated_measures_anova_sd(self) -> Dict:
        """Run repeated measures ANOVA for stride length variability with covariates using pingouin."""
        # Prepare long-format data
        long_rows = []
        
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if '_sd_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'stride_length_variability': row[col],
                                'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna()
        
        if df_long.empty:
            return {'error': 'No valid stride length variability data for analysis'}
        
        results = {}
        
        # Basic repeated measures ANOVA (without covariates)
        try:
            # Main effect of trial type
            if len(df_long['trial_type'].unique()) > 1:
                aov_trial = pg.rm_anova(data=df_long, dv='stride_length_variability', 
                                       within='trial_type', subject='subject', detailed=True)
                results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size': float(aov_trial['ng2'].iloc[0])
                }
            
            # Add descriptive statistics
            results['descriptive_stats'] = {
                'n_subjects': len(df_long['subject'].unique()),
                'n_observations': len(df_long),
                'stride_variability_overall': float(df_long['stride_length_variability'].mean()),
                'std_stride_variability_overall': float(df_long['stride_length_variability'].std()),
                'by_trial_type': df_long.groupby('trial_type')['stride_length_variability'].agg(['mean', 'std', 'count']).to_dict(),
                'by_condition': df_long.groupby('condition')['stride_length_variability'].agg(['mean', 'std', 'count']).to_dict()
            }
            
        except Exception as e:
            results['basic_anova_error'] = str(e)
        
        return results
    
    def run_age_stratified_anova(self, age_groups: Dict[str, List[float]] = None, 
                                  min_subjects_per_group: int = 1) -> Dict:
        """Run ANOVA analysis stratified by age groups with flexible grouping options."""
        
        # Default age groups if none provided
        if age_groups is None:
            age_groups = {
                'younger': [7, 12],    # 7-12 years
                'middle': [12, 15],    # 12-15 years  
                'older': [15, 18]      # 15-18 years
            }
        
        # Prepare long-format data
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            age = row.get('age', np.nan)
            
            if pd.isna(age):
                continue
                
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'success_rate': row[col],
                                'age': age,
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['age', 'success_rate'])
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {
            'age_group_definitions': age_groups,
            'min_subjects_threshold': min_subjects_per_group,
            'total_subjects_analyzed': len(df_long['subject'].unique()),
            'age_range': [float(df_long['age'].min()), float(df_long['age'].max())],
            'group_analyses': {},
            'between_group_comparisons': {},
            'summary_comparison': {}
        }
        
        # Assign subjects to age groups
        df_long['age_group'] = None
        group_assignments = {}
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_long['age'] >= min_age) & (df_long['age'] < max_age)
            df_long.loc[mask, 'age_group'] = group_name
            
            # Track which subjects are in each group
            subjects_in_group = df_long[mask]['subject'].unique()
            group_assignments[group_name] = {
                'subjects': list(subjects_in_group),
                'n_subjects': len(subjects_in_group),
                'age_range': [float(df_long[mask]['age'].min()) if len(subjects_in_group) > 0 else np.nan,
                             float(df_long[mask]['age'].max()) if len(subjects_in_group) > 0 else np.nan],
                'mean_age': float(df_long[mask]['age'].mean()) if len(subjects_in_group) > 0 else np.nan
            }
        
        results['group_assignments'] = group_assignments
        
        # Run ANOVA for each age group
        for group_name, group_info in group_assignments.items():
            if group_info['n_subjects'] < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={group_info["n_subjects"]}, minimum={min_subjects_per_group})'
                }
                continue
            
            group_data = df_long[df_long['age_group'] == group_name].copy()
            group_results = {}
            
            try:
                # Basic repeated measures ANOVA
                aov_trial = pg.rm_anova(data=group_data, dv='success_rate', 
                                       within='trial_type', subject='subject', detailed=True)
                group_results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size_eta2': float(aov_trial['ng2'].iloc[0]),
                    'significant': float(aov_trial['p-unc'].iloc[0]) < 0.05
                }
                
            except Exception as e:
                group_results['anova_error'] = str(e)
            
            # Descriptive statistics
            try:
                group_results['mean_success_rates'] = {
                    trial: float(group_data[group_data['trial_type'] == trial]['success_rate'].mean())
                    for trial in group_data['trial_type'].unique()
                }
                
                group_results['descriptive_stats'] = {
                    'n_subjects': group_info['n_subjects'],
                    'n_observations': len(group_data),
                    'age_info': {
                        'mean_age': group_info['mean_age'],
                        'age_range': group_info['age_range']
                    }
                }
                
            except Exception as e:
                group_results['descriptive_error'] = str(e)
            
            results['group_analyses'][group_name] = group_results
        
        return results
    
    def _interpret_cohens_d(self, d):
        """Interpret Cohen's d effect size."""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "negligible"
        elif abs_d < 0.5:
            return "small"
        elif abs_d < 0.8:
            return "medium"
        else:
            return "large"
    
    def run_correlation_analysis(self) -> Dict:
        """Run correlation analysis between key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols)
        
        # Calculate correlation matrix
        corr_data = self.filtered_df[key_cols].corr()
        
        # Extract significant correlations
        significant_corrs = {}
        for i, col1 in enumerate(key_cols):
            for j, col2 in enumerate(key_cols):
                if i < j:  # Avoid duplicates
                    valid_data = self.filtered_df[[col1, col2]].dropna()
                    if len(valid_data) > 5:
                        r, p = pearsonr(valid_data[col1], valid_data[col2])
                        if p < 0.05:
                            significant_corrs[f'{col1}_vs_{col2}'] = {
                                'r': r,
                                'p': p,
                                'n': len(valid_data)
                            }
        
        return {
            'correlation_matrix': corr_data.to_dict(),
            'significant_correlations': significant_corrs
        }


# ==============================================================================
# PART 10: SUCCESS MESSAGE
# ==============================================================================

print("✅ DataManager and StatisticalAnalyzer classes loaded successfully!")
print("📊 All statistical methods properly structured and indented.")
print("🔧 Ready to integrate with visualization and main analysis classes.")

# Motor Learning Analysis Pipeline - Final Parts
# This completes the visualization and main analysis classes

# ==============================================================================
# PART 11: FLEXIBLE PLOTTING SYSTEM
# ==============================================================================

class PlotType(Enum):
    """Enumeration of available plot types."""
    SCATTER = "scatter"
    LINE = "line"
    BOX = "box"
    VIOLIN = "violin"
    HISTOGRAM = "histogram"
    BAR = "bar"
    HEATMAP = "heatmap"
    CONNECTED_PAIRS = "connected_pairs"


class EnhancedPlottingRegistry:
    """Enhanced registry that automatically includes all plotting methods."""
    
    def __init__(self):
        self._methods = {}
        self._builtin_methods = {}
        self._custom_methods = {}
        self._method_sources = {}  # Track where methods come from
    
    def register(self, name: str, method: Callable, source: str = 'custom'):
        """Register a new plotting method."""
        self._methods[name] = method
        self._method_sources[name] = source
        
        if source == 'builtin':
            self._builtin_methods[name] = method
        else:
            self._custom_methods[name] = method
    
    def get_method(self, name: str) -> Optional[Callable]:
        """Get a registered plotting method."""
        return self._methods.get(name)
    
    def list_methods(self) -> List[str]:
        """List all registered methods."""
        return list(self._methods.keys())
    
    def list_builtin_methods(self) -> List[str]:
        """List only built-in plotting methods."""
        return list(self._builtin_methods.keys())
    
    def list_custom_methods(self) -> List[str]:
        """List only custom plotting methods."""
        return list(self._custom_methods.keys())
    
    def get_method_info(self, name: str) -> Dict[str, Any]:
        """Get detailed information about a method."""
        method = self.get_method(name)
        if not method:
            return {}
        
        return {
            'name': name,
            'function': method,
            'source': self._method_sources.get(name, 'unknown'),
            'docstring': method.__doc__ or 'No documentation available',
            'signature': str(inspect.signature(method)) if hasattr(inspect, 'signature') else 'N/A',
            'module': getattr(method, '__module__', 'unknown')
        }
    
    def search_methods(self, keyword: str) -> List[str]:
        """Search for methods containing a keyword."""
        return [name for name in self._methods.keys() if keyword.lower() in name.lower()]
    
    def get_methods_by_source(self, source: str) -> List[str]:
        """Get methods from a specific source (builtin, custom, etc.)."""
        return [name for name, src in self._method_sources.items() if src == source]


# Create the enhanced global registry
PLOT_REGISTRY = EnhancedPlottingRegistry()


def register_plot_method(name: str):
    """Decorator to register new plotting methods."""
    def decorator(func):
        PLOT_REGISTRY.register(name, func)
        return func
    return decorator


class FlexiblePlotter(BaseVisualizer):
    """Enhanced base class with dynamic axis control and extensible methods."""
    
    def __init__(self, data: pd.DataFrame, config: AnalysisConfig):
        super().__init__(config)
        self.data = data
        
        # Apply motor noise filter if column exists
        if 'mot_noise' in data.columns:
            self.filtered_data = data[data['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_data = data
    
    def get_available_columns(self, pattern: str = None) -> List[str]:
        """Get available columns, optionally filtered by pattern."""
        cols = self.filtered_data.columns.tolist()
        if pattern:
            cols = [col for col in cols if pattern in col]
        return cols
    
    def get_trial_columns(self, metric: str, trials: List[str] = None, 
                         conditions: List[str] = None) -> List[str]:
        """Get columns for specific metric across trials and conditions."""
        if trials is None:
            trials = ['vis1', 'invis', 'vis2']
        if conditions is None:
            conditions = ['max', 'min']
        
        cols = []
        for trial in trials:
            for condition in conditions:
                col = f'{trial}_{metric}_{condition}_const'
                if col in self.filtered_data.columns:
                    cols.append(col)
        return cols
    
    def create_figure(self, size_type: str = 'medium', **kwargs) -> Tuple[plt.Figure, plt.Axes]:
        """Create figure with config-based sizing."""
        size_map = {
            'small': self.plot_config.figure_size_small,
            'medium': self.plot_config.figure_size_medium,
            'large': self.plot_config.figure_size_large,
            'wide': self.plot_config.figure_size_wide
        }
        
        figsize = size_map.get(size_type, self.plot_config.figure_size_medium)
        fig, ax = plt.subplots(figsize=figsize, **kwargs)
        return fig, ax
    
    def create_subplots(self, nrows: int, ncols: int, size_type: str = 'large', 
                       **kwargs) -> Tuple[plt.Figure, np.ndarray]:
        """Create subplots with config-based sizing and spacing."""
        size_map = {
            'small': self.plot_config.figure_size_small,
            'medium': self.plot_config.figure_size_medium,
            'large': self.plot_config.figure_size_large,
            'wide': self.plot_config.figure_size_wide
        }
        
        base_size = size_map.get(size_type, self.plot_config.figure_size_large)
        figsize = (base_size[0] * ncols / 2, base_size[1] * nrows / 2)
        
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize,
                                gridspec_kw={
                                    'wspace': self.plot_config.subplot_wspace,
                                    'hspace': self.plot_config.subplot_hspace
                                },
                                **kwargs)
        return fig, axes
    
    def apply_styling(self, ax: plt.Axes, title: str = "", 
                     xlabel: str = "", ylabel: str = "", **kwargs):
        """Apply styling to axes with config-based settings."""
        if title:
            ax.set_title(title, fontsize=self.plot_config.font_size_title, 
                        fontweight=self.plot_config.font_weight_bold)
        if xlabel:
            ax.set_xlabel(xlabel, fontsize=self.plot_config.font_size_medium)
        if ylabel:
            ax.set_ylabel(ylabel, fontsize=self.plot_config.font_size_medium)
        
        ax.grid(True, alpha=self.plot_config.grid_alpha)
        
        # Apply any additional styling
        for key, value in kwargs.items():
            if hasattr(ax, f'set_{key}'):
                getattr(ax, f'set_{key}')(value)
    
    def plot_flexible_scatter(self, x_var: str, y_var: str, 
                             color_var: str = None,
                             group_var: str = None,
                             size_var: str = None,
                             filename: str = None,
                             title: str = None,
                             add_trendline: bool = True,
                             colormap: str = 'plasma',
                             size_type: str = 'medium',
                             **kwargs) -> Optional[Path]:
        """Flexible scatter plot with dynamic axis assignment."""
        
        # Validate columns exist
        required_cols = [x_var, y_var]
        if color_var:
            required_cols.append(color_var)
        if group_var:
            required_cols.append(group_var)
        if size_var:
            required_cols.append(size_var)
        
        missing_cols = [col for col in required_cols if col not in self.filtered_data.columns]
        if missing_cols:
            print(f"Missing columns: {missing_cols}")
            return None
        
        # Get valid data
        plot_data = self.filtered_data[required_cols].dropna()
        if plot_data.empty:
            print("No valid data for plotting")
            return None
        
        # Create figure
        fig, ax = self.create_figure(size_type)
        
        # Handle grouping
        if group_var:
            groups = plot_data[group_var].unique()
            colors = [self.plot_config.colors.get(str(group), self.plot_config.colors['primary']) 
                     for group in groups]
            
            for i, group in enumerate(groups):
                group_data = plot_data[plot_data[group_var] == group]
                
                # Set up scatter parameters
                scatter_kwargs = {
                    'alpha': kwargs.get('alpha', self.plot_config.alpha_scatter),
                    'label': str(group),
                    'color': colors[i % len(colors)]
                }
                
                if size_var:
                    scatter_kwargs['s'] = group_data[size_var] * kwargs.get('size_scale', 1)
                else:
                    scatter_kwargs['s'] = kwargs.get('s', self.plot_config.marker_size_medium)
                
                scatter_kwargs.update({k: v for k, v in kwargs.items() 
                                     if k not in ['alpha', 's', 'size_scale']})
                
                ax.scatter(group_data[x_var], group_data[y_var], **scatter_kwargs)
            
            ax.legend()
        
        else:
            # Single group plotting
            scatter_kwargs = {
                'alpha': kwargs.get('alpha', self.plot_config.alpha_scatter),
                'edgecolors': 'white',
                'linewidths': 0.5
            }
            
            if color_var:
                scatter_kwargs['c'] = plot_data[color_var]
                scatter_kwargs['cmap'] = colormap
                scatter = ax.scatter(plot_data[x_var], plot_data[y_var], **scatter_kwargs)
                plt.colorbar(scatter, ax=ax, label=color_var)
            else:
                scatter_kwargs['color'] = kwargs.get('color', self.plot_config.colors['primary'])
                ax.scatter(plot_data[x_var], plot_data[y_var], **scatter_kwargs)
            
            if size_var:
                scatter_kwargs['s'] = plot_data[size_var] * kwargs.get('size_scale', 1)
            else:
                scatter_kwargs['s'] = kwargs.get('s', self.plot_config.marker_size_medium)
        
        # Add trendline if requested
        if add_trendline:
            self.add_trendline(ax, plot_data[x_var], plot_data[y_var])
        
        # Apply styling
        plot_title = title or f'{y_var} vs {x_var}'
        self.apply_styling(ax, title=plot_title, xlabel=x_var, ylabel=y_var)
        
        plt.tight_layout()
        
        # Save if filename provided
        if filename:
            return self.save_figure(fig, filename, 'population')
        else:
            plt.show()
            return None


# ==============================================================================
# PART 12: POPULATION VISUALIZER
# ==============================================================================

class PopulationVisualizer(FlexiblePlotter):
    """Population-level plotting with specialized methods."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        super().__init__(metrics_df, config)
        self.metrics_df = metrics_df
    
    def get_age_group_color(self, index: int) -> str:
        """Get color for age group by index."""
        colors = self.plot_config.age_group_colors
        return colors[index % len(colors)]
    
    def plot_age_stratified_results(self, age_results: Dict, filename: str, **kwargs) -> Optional[Path]:
        """Create comprehensive age-stratified ANOVA visualization."""
        
        # Extract valid groups
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'trial_type_effect' in results]
        
        if not valid_groups:
            print("No valid age groups found for visualization")
            return None
        
        # Set up the figure with enhanced layout
        n_groups = len(valid_groups)
        fig = plt.figure(figsize=(20, 16))
        
        # Create subplot grid: 3 rows with flexible columns
        gs = fig.add_gridspec(3, max(n_groups, 3), hspace=0.4, wspace=0.3, 
                             height_ratios=[1, 1, 1.2])
        
        # Color scheme for trial types
        trial_colors = {
            'vis1': self.plot_config.colors['vis1'], 
            'invis': self.plot_config.colors['invis'], 
            'vis2': self.plot_config.colors['vis2']
        }
        
        # Row 1: Mean success rates by age group and trial type
        for i, group in enumerate(valid_groups):
            ax = fig.add_subplot(gs[0, i])
            group_data = age_results['group_analyses'][group]
            
            if 'mean_success_rates' in group_data:
                trials = list(group_data['mean_success_rates'].keys())
                means = list(group_data['mean_success_rates'].values())
                colors = [trial_colors.get(trial, self.plot_config.colors['primary']) for trial in trials]
                
                bars = ax.bar(trials, means, color=colors, 
                             alpha=self.plot_config.alpha_fill, 
                             edgecolor='black', linewidth=1)
                
                # Add value labels on bars
                for bar, mean in zip(bars, means):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{mean:.2f}', ha='center', va='bottom', 
                           fontweight=self.plot_config.font_weight_bold,
                           fontsize=self.plot_config.font_size_small)
                
                ax.set_ylim(0, 1)
                self.apply_styling(
                    ax,
                    title=f'{group.title()}\\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})',
                    ylabel='Mean Success Rate' if i == 0 else ''
                )
                
                # Add age info
                age_info = group_data.get('descriptive_stats', {}).get('age_info', {})
                if 'mean_age' in age_info:
                    ax.text(0.5, 0.95, f'Mean age: {age_info["mean_age"]:.1f}y', 
                           transform=ax.transAxes, ha='center', va='top', 
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7),
                           fontsize=self.plot_config.font_size_small)
        
        # Row 2: Effect sizes (eta-squared) comparison
        ax_effect = fig.add_subplot(gs[1, :n_groups])
        
        effect_sizes = []
        group_names = []
        significance = []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                group_names.append(group.title())
                significance.append(group_data['trial_type_effect']['significant'])
        
        if effect_sizes:
            colors = [self.plot_config.colors['success'] if sig else self.plot_config.colors['danger'] 
                     for sig in significance]
            bars = ax_effect.bar(group_names, effect_sizes, color=colors, 
                               alpha=self.plot_config.alpha_fill, edgecolor='black')
            
            # Add significance indicators
            for i, (bar, sig, eta2) in enumerate(zip(bars, significance, effect_sizes)):
                height = bar.get_height()
                sig_text = '***' if sig else 'ns'
                ax_effect.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                              f'{eta2:.3f}\\n{sig_text}', ha='center', va='bottom', 
                              fontweight=self.plot_config.font_weight_bold,
                              fontsize=self.plot_config.font_size_small)
            
            self.apply_styling(
                ax_effect,
                title='Trial Type Effect Sizes by Age Group',
                ylabel='Effect Size (η²)'
            )
            
            # Add effect size interpretation lines
            ax_effect.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='Small (0.01)')
            ax_effect.axhline(y=0.06, color='orange', linestyle='--', alpha=0.5, label='Medium (0.06)')
            ax_effect.axhline(y=0.14, color='red', linestyle='--', alpha=0.5, label='Large (0.14)')
            ax_effect.legend(loc='upper right', fontsize=self.plot_config.font_size_small)
        
        plt.suptitle('Age-Stratified ANOVA Results: Trial Type Effects', 
                    fontsize=self.plot_config.font_size_suptitle,
                    fontweight=self.plot_config.font_weight_bold)
        
        return self.save_figure(fig, filename, 'population')
    
    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_data.columns
        
        # Calculate age limits (consistent across all plots)
        age_data = self.filtered_data['age'].dropna()
        if not age_data.empty:
            age_min = age_data.min()
            age_max = age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = self.create_subplots(len(trials), len(conditions), 'large')
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_sr_{condition}_const'
                
                if col in self.filtered_data.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_data[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Success Rate')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                
                # Set consistent age axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
        
        plt.suptitle('Age vs Success Rates by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig,'age_vs_success_rates.png', 'population')
    
    def plot_correlation_matrix(self) -> Path:
        """Plot correlation matrix of key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_data.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_data.columns:
            key_cols.append('pref_asymmetry')
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_data.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols[:6])  # Limit to first 6 to avoid clutter
        
        if len(key_cols) > 1:
            fig = plt.figure(figsize=(12, 10))
            
            corr_matrix = self.filtered_data[key_cols].corr()
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                       square=True, linewidths=0.5, fmt='.2f')
            
            plt.title('Correlation Matrix of Key Variables', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            return self.save_figure(fig, 'correlation_matrix.png', 'population')


# ==============================================================================
# PART 13: INDIVIDUAL VISUALIZER
# ==============================================================================

class IndividualVisualizer(FlexiblePlotter):
    """Individual subject plotting with specialized methods."""
    
    def __init__(self, data: pd.DataFrame, data_manager, config: AnalysisConfig):
        super().__init__(data, config)
        self.data_manager = data_manager
    
    def plot_subject_overview(self, subject_id: str, 
                             metrics: List[str] = None,
                             filename: str = None,
                             **kwargs) -> Optional[Path]:
        """Flexible subject overview plot with customizable metrics."""
        
        subject = self.data_manager.subjects.get(subject_id)
        if not subject:
            print(f"Subject {subject_id} not found")
            return None
        
        if metrics is None:
            metrics = ['success_rates', 'motor_noise', 'age_comparison', 'subject_info']
        
        # Create subplot grid based on number of metrics
        n_metrics = len(metrics)
        if n_metrics <= 2:
            nrows, ncols = 1, n_metrics
        elif n_metrics <= 4:
            nrows, ncols = 2, 2
        else:
            nrows, ncols = 2, 3  # Maximum 6 plots
        
        fig, axes = self.create_subplots(nrows, ncols, 'large')
        if n_metrics == 1:
            axes = [axes]
        elif nrows == 1:
            axes = axes.reshape(1, -1)
        
        for i, metric in enumerate(metrics[:6]):  # Limit to 6 plots
            if i >= len(axes.flat):
                break
                
            ax = axes.flat[i]
            
            if metric == 'success_rates':
                self._plot_subject_success_rates(ax, subject)
            elif metric == 'motor_noise':
                self._plot_subject_motor_noise(ax, subject)
            elif metric == 'age_comparison':
                self._plot_subject_age_comparison(ax, subject)
            elif metric == 'subject_info':
                self._plot_subject_info(ax, subject)
            else:
                # Try to plot custom metric
                if hasattr(self, f'_plot_subject_{metric}'):
                    getattr(self, f'_plot_subject_{metric}')(ax, subject)
                else:
                    ax.text(0.5, 0.5, f'Unknown metric: {metric}', 
                           ha='center', va='center', transform=ax.transAxes)
        
        # Hide unused subplots
        for i in range(len(metrics), len(axes.flat)):
            axes.flat[i].set_visible(False)
        
        plt.suptitle(f'Subject {subject_id} Overview',
                    fontsize=self.plot_config.font_size_suptitle,
                    fontweight=self.plot_config.font_weight_bold)
        plt.tight_layout()
        
        if filename:
            return self.save_figure(fig, filename, 'individual')
        else:
            plt.show()
            return None
    
    def _plot_subject_success_rates(self, ax: plt.Axes, subject):
        """Plot success rates for a subject."""
        trial_types = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        x_pos = np.arange(len(trial_types))
        width = 0.35
        
        max_rates = []
        min_rates = []
        
        for trial in trial_types:
            trial_dict = subject.trial_data.get(trial)
            if trial_dict and trial_dict['data'] is not None:
                df = trial_dict['data']
                
                # Get success rates for both conditions
                max_data, _ = self.data_manager._get_period_data(df, 'max')
                min_data, _ = self.data_manager._get_period_data(df, 'min')
                
                max_rate = max_data['Success'].mean() if max_data is not None else 0
                min_rate = min_data['Success'].mean() if min_data is not None else 0
                
                max_rates.append(max_rate)
                min_rates.append(min_rate)
            else:
                max_rates.append(0)
                min_rates.append(0)
        
        ax.bar(x_pos - width/2, max_rates, width, 
               label='Max Target', alpha=self.plot_config.alpha_fill,
               color=self.plot_config.colors['max_target'])
        ax.bar(x_pos + width/2, min_rates, width, 
               label='Min Target', alpha=self.plot_config.alpha_fill,
               color=self.plot_config.colors['min_target'])
        
        self.apply_styling(
            ax,
            title='Success Rates by Trial Type',
            xlabel='Trial Type',
            ylabel='Success Rate'
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels([t.upper() for t in trial_types])
        ax.legend()
    
    def _plot_subject_motor_noise(self, ax: plt.Axes, subject):
        """Plot motor noise visualization from preference trial."""
        pref_dict = subject.trial_data.get('pref')
        if pref_dict and pref_dict['data'] is not None:
            pref_df = pref_dict['data']
            if all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
                right_steps = pref_df['Right step length']
                left_steps = pref_df['Left step length']
                
                ax.plot(right_steps, label='Right Steps', 
                       alpha=self.plot_config.alpha_line,
                       color=self.plot_config.colors['primary'],
                       linewidth=self.plot_config.line_width_medium)
                ax.plot(left_steps, label='Left Steps', 
                       alpha=self.plot_config.alpha_line,
                       color=self.plot_config.colors['secondary'],
                       linewidth=self.plot_config.line_width_medium)
                
                self.apply_styling(
                    ax,
                    title='Preference Trial Step Lengths',
                    xlabel='Stride Number',
                    ylabel='Step Length'
                )
                ax.legend()
            else:
                ax.text(0.5, 0.5, 'Missing Step Length Data', ha='center', va='center',
                       transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, 'No Preference Data', ha='center', va='center',
                   transform=ax.transAxes, 
                   fontsize=self.plot_config.font_size_medium)
        ax.set_title('Preference Trial Step Lengths')
    
    def _plot_subject_age_comparison(self, ax: plt.Axes, subject):
        """Plot age comparison against population."""
        all_ages = [s.age for s in self.data_manager.subjects.values()]
        ax.hist(all_ages, bins=20, alpha=self.plot_config.alpha_fill, 
               label='All Subjects', color=self.plot_config.colors['primary'])
        ax.axvline(subject.age, color=self.plot_config.colors['danger'], 
                  linestyle='--', linewidth=self.plot_config.line_width_thick,
                  label=f'This Subject (Age: {subject.age:.1f})')
        
        self.apply_styling(
            ax,
            title='Age Distribution',
            xlabel='Age (years)',
            ylabel='Frequency'
        )
        ax.legend()
    
    def _plot_subject_info(self, ax: plt.Axes, subject):
        """Plot subject information panel."""
        info_text = (f'Subject ID: {subject.subject_id}\\n'
                    f'Age: {subject.age:.1f} years\\n'
                    f'Session: {subject.metadata.get("Session Date", "Unknown")}')
        
        ax.text(0.5, 0.5, info_text, ha='center', va='center', 
               transform=ax.transAxes, 
               fontsize=self.plot_config.font_size_medium,
               bbox=dict(boxstyle='round', facecolor='lightblue', 
                        alpha=self.plot_config.alpha_fill))
        ax.set_title('Subject Information')
        ax.axis('off')


# ==============================================================================
# PART 14: VISUALIZATION COORDINATOR
# ==============================================================================

class VisualizationCoordinator:
    """Enhanced coordinator with flexible plotting capabilities."""
    
    def __init__(self, metrics_df, data_manager, config: AnalysisConfig):
        self.config = config
        self.metrics_df = metrics_df
        self.data_manager = data_manager
        
        # Initialize flexible plotters
        self.population_plotter = PopulationVisualizer(metrics_df, config)
        self.individual_plotter = IndividualVisualizer(metrics_df, data_manager, config)
    
    def update_config(self, **kwargs):
        """Update configuration across all plotters."""
        self.config.update_plot_config(**kwargs)
        
        # Update all plotter configs
        for plotter in [self.population_plotter, self.individual_plotter]:
            plotter.plot_config = self.config.plot_config
            plotter._setup_matplotlib_style()
    
    def create_standard_plots(self) -> List[Path]:
        """Create standard set of plots using flexible methods."""
        plot_paths = []
        
        # Age vs success rates
        try:
            path = self.population_plotter.plot_age_vs_success_rates()
            if path:
                plot_paths.append(path)
        except Exception as e:
            print(f"Failed to create age vs success rates plot: {e}")
        
        # Correlation matrix
        try:
            path = self.population_plotter.plot_correlation_matrix()
            if path:
                plot_paths.append(path)
        except Exception as e:
            print(f"Failed to create correlation matrix: {e}")
        
        return plot_paths
    
    def create_custom_plots(self, custom_methods: List[Dict]) -> List[Path]:
        """
        Create custom plots using flexible methods.
        
        Parameters:
        -----------
        custom_methods : List[Dict]
            List of method specifications, each containing:
            - 'method': method name
            - 'args': positional arguments
            - 'kwargs': keyword arguments
        """
        plot_paths = []
        
        for method_spec in custom_methods:
            method_name = method_spec['method']
            args = method_spec.get('args', [])
            kwargs = method_spec.get('kwargs', {})
            
            try:
                # Check if it's a built-in method
                if hasattr(self.population_plotter, method_name):
                    method = getattr(self.population_plotter, method_name)
                    path = method(*args, **kwargs)
                # Check if it's a registered method
                elif hasattr(self.population_plotter, 'call_registered_method'):
                    path = self.population_plotter.call_registered_method(
                        method_name, *args, **kwargs)
                else:
                    print(f"Unknown method: {method_name}")
                    continue
                
                if path:
                    plot_paths.append(path)
                    
            except Exception as e:
                print(f"Failed to create custom plot '{method_name}': {e}")
        
        return plot_paths
    
    def create_individual_plots(self, subject_ids: List[str], 
                              custom_metrics: List[str] = None) -> List[Path]:
        """Create individual subject plots with flexible metrics."""
        plot_paths = []
        
        for subject_id in subject_ids:
            try:
                path = self.individual_plotter.plot_subject_overview(
                    subject_id=subject_id,
                    metrics=custom_metrics,
                    filename=f'subject_overview_{subject_id}.png'
                )
                if path:
                    plot_paths.append(path)
            except Exception as e:
                print(f"Failed to create plot for subject {subject_id}: {e}")
        
        return plot_paths


# ==============================================================================
# PART 15: MAIN ANALYSIS INTERFACE
# ==============================================================================

class MotorLearningAnalysis:
    """Main analysis interface that coordinates all components."""
    
    def __init__(self, config: AnalysisConfig = None):
        self.config = config or AnalysisConfig()
        self.data_manager: Optional[DataManager] = None
        self.metrics_df: Optional[pd.DataFrame] = None
        self.statistical_analyzer: Optional[StatisticalAnalyzer] = None
        self.population_visualizer: Optional[PopulationVisualizer] = None
        self.individual_visualizer: Optional[IndividualVisualizer] = None
        self.visualization_coordinator: Optional[VisualizationCoordinator] = None
        self.results: Optional[Dict] = None
    
    def load_data(self, metadata_path: str, data_root_dir: str, 
                  force_reprocess: bool = False) -> 'MotorLearningAnalysis':
        """Load and process data."""
        self.data_manager = DataManager(
            metadata_path, data_root_dir, 
            self.config, force_reprocess
        )
        return self
    
    def filter_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter data based on criteria."""
        if self.data_manager:
            self.data_manager = self.data_manager.filter_subjects(**kwargs)
        return self
    
    def calculate_metrics(self) -> 'MotorLearningAnalysis':
        """Calculate metrics for all subjects."""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        self.metrics_df = self.data_manager.calculate_metrics()
        
        # Initialize analyzers with metrics  
        self.statistical_analyzer = StatisticalAnalyzer(self.metrics_df, self.config)
        self.population_visualizer = PopulationVisualizer(self.metrics_df, self.config)
        self.individual_visualizer = IndividualVisualizer(self.metrics_df, self.data_manager, self.config)
        self.visualization_coordinator = VisualizationCoordinator(
            self.metrics_df, self.data_manager, self.config)
        
        return self
    
    def update_plot_config(self, **kwargs) -> 'MotorLearningAnalysis':
        """Update plotting configuration after analysis creation."""
        self.config.update_plot_config(**kwargs)
        
        # Update visualization coordinator if it exists
        if self.visualization_coordinator:
            self.visualization_coordinator.update_config(**kwargs)
        
        return self
    
    def update_colors(self, color_dict: Dict[str, str]) -> 'MotorLearningAnalysis':
        """Update color scheme."""
        self.config.plot_config.update_colors(color_dict)
        
        if self.visualization_coordinator:
            self.visualization_coordinator.update_config()
        
        return self
    
    def get_plotter(self, plotter_type: str = 'population'):
        """Get specific plotter for direct access."""
        if not self.visualization_coordinator:
            raise ValueError("Visualization coordinator not initialized. Call calculate_metrics() first.")
        
        if plotter_type == 'population':
            return self.visualization_coordinator.population_plotter
        elif plotter_type == 'individual':
            return self.visualization_coordinator.individual_plotter
        else:
            raise ValueError(f"Unknown plotter type: {plotter_type}")
    
    def run_analysis(self, include_visualizations: bool = True,
                    create_individual_plots: bool = True,
                    max_individual_plots: int = 5,
                    custom_plots: List[Dict] = None,
                    create_age_stratified: bool = False,
                    age_groups: Dict[str, List[float]] = None) -> 'MotorLearningAnalysis':
        """
        Run comprehensive analysis with flexible visualization options.
        
        Parameters:
        -----------
        include_visualizations : bool
            Whether to create standard visualizations
        create_individual_plots : bool
            Whether to create individual subject plots
        max_individual_plots : int
            Maximum number of individual plots to create
        custom_plots : List[Dict], optional
            Custom plot specifications
        create_age_stratified : bool
            Whether to create age-stratified ANOVA plots
        age_groups : Dict[str, List[float]], optional
            Custom age groups for stratified analysis
        """
        if self.metrics_df is None:
            raise ValueError("Metrics not calculated. Call calculate_metrics() first.")
        
        self.results = {
            'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'n_subjects': len(self.metrics_df),
            'analyses': {},
            'config_snapshot': {
                'motor_noise_threshold': self.config.motor_noise_threshold,
                'figure_dpi': self.config.plot_config.figure_dpi,
                'colors': dict(self.config.plot_config.colors)
            }
        }
        
        # Statistical analyses
        if self.statistical_analyzer:
            print("Running statistical analyses...")
            
            try:
                self.results['analyses']['regression'] = self.statistical_analyzer.run_regression_analysis()
            except Exception as e:
                self.results['analyses']['regression'] = {'error': str(e)}
            
            try:
                self.results['analyses']['anova_success_rates'] = self.statistical_analyzer.run_repeated_measures_anova()
            except Exception as e:
                self.results['analyses']['anova_success_rates'] = {'error': str(e)}
            
            try:
                self.results['analyses']['anova_mean_stride_length'] = self.statistical_analyzer.run_repeated_measures_anova_msl()
            except Exception as e:
                self.results['analyses']['anova_mean_stride_length'] = {'error': str(e)}
            
            try:
                self.results['analyses']['anova_stride_variability'] = self.statistical_analyzer.run_repeated_measures_anova_sd()
            except Exception as e:
                self.results['analyses']['anova_stride_variability'] = {'error': str(e)}
            
            try:
                self.results['analyses']['correlations'] = self.statistical_analyzer.run_correlation_analysis()
            except Exception as e:
                self.results['analyses']['correlations'] = {'error': str(e)}
            
            # Age-stratified analysis if requested
            if create_age_stratified:
                try:
                    self.results['analyses']['age_stratified'] = self.statistical_analyzer.run_age_stratified_anova(
                        age_groups=age_groups
                    )
                    print("Completed age-stratified ANOVA analysis")
                except Exception as e:
                    self.results['analyses']['age_stratified'] = {'error': str(e)}
                    print(f"Failed age-stratified analysis: {e}")
        
        # Enhanced visualizations
        if include_visualizations and self.visualization_coordinator:
            print("Creating visualizations...")
            
            self.results['visualizations'] = {
                'standard': [],
                'custom': [],
                'individual': [],
                'age_stratified': [],
                'config_used': {
                    'figure_dpi': self.config.plot_config.figure_dpi,
                    'colors': dict(self.config.plot_config.colors),
                    'figure_sizes': {
                        'small': self.config.plot_config.figure_size_small,
                        'medium': self.config.plot_config.figure_size_medium,
                        'large': self.config.plot_config.figure_size_large,
                        'wide': self.config.plot_config.figure_size_wide
                    }
                }
            }
            
            # Standard plots
            try:
                standard_paths = self.visualization_coordinator.create_standard_plots()
                self.results['visualizations']['standard'] = [str(p) for p in standard_paths]
                print(f"Created {len(standard_paths)} standard plots")
            except Exception as e:
                print(f"Failed to create standard plots: {e}")
                self.results['visualizations']['standard_error'] = str(e)
            
            # Age-stratified plots
            if create_age_stratified:
                try:
                    age_plot_paths = self.create_age_stratified_plots(age_groups=age_groups)
                    self.results['visualizations']['age_stratified'] = [str(p) for p in age_plot_paths.values()]
                    print(f"Created {len(age_plot_paths)} age-stratified plots")
                except Exception as e:
                    print(f"Failed to create age-stratified plots: {e}")
                    self.results['visualizations']['age_stratified_error'] = str(e)
            
            # Custom plots
            if custom_plots:
                try:
                    custom_paths = self.visualization_coordinator.create_custom_plots(custom_plots)
                    self.results['visualizations']['custom'] = [str(p) for p in custom_paths]
                    print(f"Created {len(custom_paths)} custom plots")
                except Exception as e:
                    print(f"Failed to create custom plots: {e}")
                    self.results['visualizations']['custom_error'] = str(e)
            
            # Individual plots
            if create_individual_plots and self.data_manager:
                try:
                    subject_ids = list(self.data_manager.subjects.keys())[:max_individual_plots]
                    individual_paths = self.visualization_coordinator.create_individual_plots(subject_ids)
                    self.results['visualizations']['individual'] = [str(p) for p in individual_paths]
                    print(f"Created {len(individual_paths)} individual plots for {len(subject_ids)} subjects")
                except Exception as e:
                    print(f"Failed to create individual plots: {e}")
                    self.results['visualizations']['individual_error'] = str(e)
        
        # Generate report
        self._generate_report()
        
        return self
    
    def create_age_stratified_plots(self, age_groups: Dict[str, List[float]] = None) -> Dict[str, Path]:
        """Create age-stratified plots."""
        if not self.statistical_analyzer:
            raise ValueError("Statistical analyzer not initialized")
        
        # Run age-stratified analysis if not already done
        age_results = self.statistical_analyzer.run_age_stratified_anova(age_groups=age_groups)
        
        plot_paths = {}
        
        # Main age-stratified results plot
        try:
            path = self.population_visualizer.plot_age_stratified_results(
                age_results, 'age_stratified_anova.png'
            )
            if path:
                plot_paths['main'] = path
        except Exception as e:
            print(f"Failed to create main age-stratified plot: {e}")
        
        return plot_paths
    
    def get_results(self) -> Dict:
        """Get analysis results."""
        return self.results
    
    def _generate_report(self):
        """Generate enhanced analysis report."""
        if not self.results:
            return
        
        report_path = self.config.reports_dir / f"analysis_report_{self.results['timestamp']}.json"
        
        # Convert to JSON-serializable format
        def make_serializable(obj):
            if isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, Path):
                return str(obj)
            elif pd.isna(obj):
                return None
            elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
                return str(type(obj).__name__)
            return obj
        
        serializable_results = json.loads(
            json.dumps(self.results, default=make_serializable)
        )
        
        with open(report_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"Enhanced report saved to: {report_path}")


# ==============================================================================
# PART 16: CONVENIENCE FUNCTIONS
# ==============================================================================

def create_custom_config(colors: Dict[str, str] = None,
                        figure_sizes: Dict[str, Tuple[int, int]] = None,
                        **kwargs) -> AnalysisConfig:
    """Create custom analysis configuration."""
    config = AnalysisConfig(**kwargs)
    
    if colors:
        config.plot_config.update_colors(colors)
    
    if figure_sizes:
        config.plot_config.update_figure_sizes(**figure_sizes)
    
    return config


def run_motor_learning_analysis(metadata_path: str, data_root_dir: str,
                               output_dir: str = 'motor_learning_output',
                               custom_colors: Dict[str, str] = None,
                               figure_dpi: int = 300,
                               required_trials: List[str] = None,
                               custom_plots: List[Dict] = None,
                               create_age_stratified: bool = False,
                               custom_age_groups: Dict[str, List[float]] = None) -> MotorLearningAnalysis:
    """Convenience function to run complete analysis with custom styling and plots."""
    
    # Create config with custom settings
    config = AnalysisConfig(base_output_dir=Path(output_dir))
    config.plot_config.figure_dpi = figure_dpi
    
    if custom_colors:
        config.plot_config.update_colors(custom_colors)
    
    # Initialize and run analysis
    analysis = (MotorLearningAnalysis(config)
                .load_data(metadata_path, data_root_dir)
                .filter_data(required_trial_types=required_trials or ['vis1', 'invis', 'vis2'])
                .calculate_metrics()
                .run_analysis(custom_plots=custom_plots,
                             create_age_stratified=create_age_stratified,
                             age_groups=custom_age_groups))
    
    return analysis


# ==============================================================================
# PART 17: EXAMPLE CUSTOM PLOT METHOD
# ==============================================================================

@register_plot_method('custom_age_heatmap')
def plot_custom_age_heatmap(plotter, metric: str = 'sr', filename: str = None, **kwargs):
    """Example custom plotting method using the registry system."""
    
    # Create age bins
    age_data = plotter.filtered_data['age'].dropna()
    age_bins = np.linspace(age_data.min(), age_data.max(), 6)
    age_labels = [f'{age_bins[i]:.1f}-{age_bins[i+1]:.1f}' for i in range(len(age_bins)-1)]
    
    # Create age groups
    plotter.filtered_data['age_group'] = pd.cut(plotter.filtered_data['age'], 
                                               bins=age_bins, labels=age_labels)
    
    # Get metric columns
    metric_cols = plotter.get_trial_columns(metric)
    if not metric_cols:
        print(f"No columns found for metric '{metric}'")
        return None
    
    # Create heatmap data
    heatmap_data = plotter.filtered_data.groupby('age_group')[metric_cols].mean()
    
    # Create figure
    fig, ax = plotter.create_figure('large')
    
    # Plot heatmap
    sns.heatmap(heatmap_data, annot=True, cmap='RdYlBu_r', ax=ax,
               fmt='.3f', cbar_kws={'label': f'{metric.upper()} Mean'})
    
    plotter.apply_styling(ax, title=f'{metric.upper()} by Age Group and Trial/Condition')
    
    plt.tight_layout()
    
    if filename:
        return plotter.save_figure(fig, filename, 'population')
    else:
        plt.show()
        return None


# ==============================================================================
# FINAL SUCCESS MESSAGE
# ==============================================================================

print("🎉 MOTOR LEARNING ANALYSIS PIPELINE COMPLETE!")
print("=" * 60)
print("✅ All classes properly structured and indented")
print("📊 Statistical analysis methods fixed") 
print("📈 Flexible visualization system included")
print("🔧 Main analysis interface ready")
print("🎨 Custom plotting methods supported")
print("=" * 60)
print("🚀 Ready to run your motor learning analysis!")

# Example usage:
"""
# Basic usage:
analysis = run_motor_learning_analysis(
    metadata_path='metadata.csv',
    data_root_dir='data/',
    output_dir='results/',
    create_age_stratified=True
)

# Advanced usage:
config = create_custom_config(
    colors={'primary': '#2E86AB', 'secondary': '#A23B72'},
    figure_dpi=400
)

analysis = (MotorLearningAnalysis(config)
    .load_data('metadata.csv', 'data/')
    .filter_data(min_age=7, max_age=18)
    .calculate_metrics()
    .run_analysis(create_age_stratified=True))
"""

✅ Motor Learning Analysis structure loaded successfully!
📝 This provides the foundation classes and configuration.
🔧 Continue by adding the DataManager and StatisticalAnalyzer classes.
✅ DataManager and StatisticalAnalyzer classes loaded successfully!
📊 All statistical methods properly structured and indented.
🔧 Ready to integrate with visualization and main analysis classes.
🎉 MOTOR LEARNING ANALYSIS PIPELINE COMPLETE!
✅ All classes properly structured and indented
📊 Statistical analysis methods fixed
📈 Flexible visualization system included
🔧 Main analysis interface ready
🎨 Custom plotting methods supported
🚀 Ready to run your motor learning analysis!


"\n# Basic usage:\nanalysis = run_motor_learning_analysis(\n    metadata_path='metadata.csv',\n    data_root_dir='data/',\n    output_dir='results/',\n    create_age_stratified=True\n)\n\n# Advanced usage:\nconfig = create_custom_config(\n    colors={'primary': '#2E86AB', 'secondary': '#A23B72'},\n    figure_dpi=400\n)\n\nanalysis = (MotorLearningAnalysis(config)\n    .load_data('metadata.csv', 'data/')\n    .filter_data(min_age=7, max_age=18)\n    .calculate_metrics()\n    .run_analysis(create_age_stratified=True))\n"

In [10]:
data_root_dir = 'muh_data/'
metadata_path = 'muh_metadata.csv'

analysis = run_motor_learning_analysis(
    metadata_path=metadata_path,
    data_root_dir=data_root_dir,
    output_dir='results/',
    create_age_stratified=True
)


✓ Loaded 110 subjects from cache
📊 Calculating metrics for 66 subjects...
✓ Successfully calculated metrics for 66 subjects
Running statistical analyses...
Completed age-stratified ANOVA analysis
Creating visualizations...
Created 2 standard plots
Created 1 age-stratified plots
Created 5 individual plots for 5 subjects
Enhanced report saved to: results\reports\analysis_report_20250808_145342.json


In [11]:
PLOT_REGISTRY.list_methods()

['custom_age_heatmap']

In [27]:
with pd.option_context('display.max_rows', None):
    print(analysis.metrics_df.ID)


0      MUH451
1      MUH731
2      MUH994
3     MUH1034
4     MUH1037
5     MUH1048
6     MUH1064
7     MUH1065
8     MUH1067
9     MUH1068
10    MUH1069
11    MUH1070
12    MUH1071
13    MUH1072
14    MUH1073
15    MUH1074
16    MUH1075
17    MUH1078
18    MUH1079
19    MUH1080
20    MUH1081
21    MUH1082
22    MUH1083
23    MUH1084
24    MUH1085
25    MUH1086
26    MUH1087
27    MUH1088
28    MUH1089
29    MUH1090
30    MUH1091
31    MUH1096
32    MUH1097
33    MUH1099
34    MUH1100
35    MUH1102
36    MUH1103
37    MUH1104
38    MUH1105
39    MUH1106
40    MUH1117
41    MUH1118
42    MUH1140
43    MUH1143
44    MUH1144
45    MUH1145
46    MUH1146
47    MUH1148
48    MUH1155
49    MUH1169
50    MUH1172
51    MUH1189
52    MUH1207
53    MUH1216
54    MUH1217
55    MUH1227
56    MUH1272
57    MUH1273
58    MUH1358
59    MUH1382
60    MUH1384
61    MUH1386
62    MUH1387
63    MUH1391
64    MUH1395
65    MUH1396
Name: ID, dtype: object
